In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
#SET BASE DIRECTORY
#This notebook expects the GSE135779 data folders (adult_individual_h5ad, GSE135779_RAW, Results, etc.) to sit one level above this Notebooks folder. Update BASE_DIR below if your data lives elsewhere.

import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path(BASE_DIR) / "Notebooks"))
from publication_utils import configure_publication_notebook

FIGURE_DIR = configure_publication_notebook(BASE_DIR, "05_GSE135779_PRIMARY_AGE_DISEASE_INTERACTION")


# Primary donor-level age-by-disease interaction analysis

This notebook fits the prespecified donor-level interaction model. Pooled expression-product rankings were removed because they are descriptive and are not used as evidence of group differences.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

cH = pd.read_csv(f"{BASE_DIR}/Results/severity_analysis/cH_per_patient_scores.csv")
cH["condition"] = "healthy"; cH["age_group"] = "pediatric"
cSLE = pd.read_csv(f"{BASE_DIR}/Results/severity_analysis/cSLE_per_patient_scores.csv")
cSLE["condition"] = "SLE"; cSLE["age_group"] = "pediatric"
aH = pd.read_csv(f"{BASE_DIR}/Results/severity_analysis/aH_per_patient_scores.csv")
aH["condition"] = "healthy"; aH["age_group"] = "adult"
aSLE = pd.read_csv(f"{BASE_DIR}/Results/severity_analysis/aSLE_per_patient_scores.csv")
aSLE["condition"] = "SLE"; aSLE["age_group"] = "adult"

combined = pd.concat([cH, cSLE, aH, aSLE], ignore_index=True)

shared = (set(cH["interaction_id"]) & set(cSLE["interaction_id"])
          & set(aH["interaction_id"]) & set(aSLE["interaction_id"]))
print(f"Interactions with per-patient scores in all four conditions: {len(shared)}")
combined = combined[combined["interaction_id"].isin(shared)]

from analysis_config import MIN_PATIENTS_PER_GROUP
MIN_PER_GROUP = MIN_PATIENTS_PER_GROUP
records = []
model_failures = []
for interaction_id, group in combined.groupby("interaction_id"):
    counts = group.groupby(["age_group", "condition"])["sample"].nunique()
    if len(counts) < 4 or (counts < MIN_PER_GROUP).any():
        continue
    if group["score"].std() == 0:
        continue
    try:
        model = smf.ols(
            "score ~ C(condition, Treatment(reference=\"healthy\")) * "
            "C(age_group, Treatment(reference=\"adult\"))", data=group
        ).fit(cov_type="HC3")
        interaction_terms = [c for c in model.pvalues.index if ":" in c]
        if not interaction_terms:
            continue
        p_interaction = model.pvalues[interaction_terms[0]]
        term = interaction_terms[0]
        coef_interaction = model.params[term]
        ci_low, ci_high = model.conf_int().loc[term]
    except Exception as exc:
        model_failures.append({"interaction_id": interaction_id, "error_type": type(exc).__name__, "message": str(exc)})
        continue

    row = group.iloc[0]
    records.append({
        "interaction_id": interaction_id,
        "source": row["source"], "target": row["target"],
        "ligand_complex": row["ligand_complex"], "receptor_complex": row["receptor_complex"],
        "interaction_coef": coef_interaction,
        "interaction_ci95_low": ci_low,
        "interaction_ci95_high": ci_high,
        "n_observations": int(model.nobs),
        "r_squared": model.rsquared,
        "condition_number": model.condition_number,
        "interaction_p_value": p_interaction,
    })

failure_columns = ["interaction_id", "error_type", "message"]
pd.DataFrame(model_failures, columns=failure_columns).to_csv(f"{BASE_DIR}/Results/severity_analysis/age_diseasestatus_model_failures.csv", index=False)

result2 = pd.DataFrame(records)
if len(result2) > 0:
    result2["fdr_q_value"] = multipletests(result2["interaction_p_value"], method="fdr_bh")[1]
    result2 = result2.sort_values("fdr_q_value")

out_path2 = f"{BASE_DIR}/Results/severity_analysis/age_diseasestatus_interaction.csv"
result2.to_csv(out_path2, index=False)

print(f"Tested {len(result2)} interactions for an age x disease-status interaction effect")
print(f"Significant at FDR < 0.05: {(result2['fdr_q_value'] < 0.05).sum() if len(result2) else 0}")
print(f"Significant at FDR < 0.10: {(result2['fdr_q_value'] < 0.10).sum() if len(result2) else 0}")
print(f"\nSaved: {out_path2}")
result2.head(20)